In [ ]:
# Standard libraries
import os

# Matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# OpenCV
import cv2

## Constants

In [ ]:
SOURCE_IMAGE_DIR = ".data/images/"
GRAYSCALE_IMAGE_DIR = ".data/eval_grayscale"

CATEGORY_LABELS = [
    "not_food",
    "italian_food",
    "japanese_food",
    "fast_food",
    "meat",
    "seafood",
    "soup",
    "salad",
    "dessert",
    "rice",
    "eggs",
]

In [ ]:
import cv2
import numpy as np

def resize_and_pad(image: cv2.Mat, target_size: tuple) -> cv2.Mat:
    height, width = image.shape[:2]
    center = np.array(image.shape[:2]) / 2
    
    # Get smallest side to determine the crop size
    crop = min(height, width)
    x = center[1] - crop / 2
    y = center[0] - crop / 2

    img_cropped = image[int(y):int(y+crop), int(x):int(x+crop)]

    # Crop the image to a square
    img_cropped = cv2.resize(img_cropped, target_size, interpolation=cv2.INTER_AREA)

    return img_cropped


## Generate Grayscale Images

In [ ]:
# CATEGORY_LABELS = ['.']
SIZE = 224
RGB_IMAGE_DIR = f".data/rgb_{SIZE}/"

total_samples = 0

for img_dir in CATEGORY_LABELS:
    sample_count = len(os.listdir(os.path.join(SOURCE_IMAGE_DIR, img_dir)))
    print(f"Processing category: {img_dir} ({sample_count} samples)")
    total_samples += sample_count

    for filename in os.listdir(os.path.join(SOURCE_IMAGE_DIR, img_dir)):
        if not (filename.endswith(".jpg") or filename.endswith(".png")): continue
        
        # Processing
        img = cv2.imread(os.path.join(SOURCE_IMAGE_DIR, img_dir, filename), cv2.IMREAD_COLOR_RGB)
        img_resized = resize_and_pad(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), target_size=(SIZE, SIZE))

        rgb_dir = os.path.join(RGB_IMAGE_DIR, f"{img_dir}")
        os.makedirs(rgb_dir, exist_ok=True)
        cv2.imwrite(os.path.join(rgb_dir, filename), img_resized)

        del img, img_resized

print("----------------------------------------")
print(f"Total samples processed: {total_samples}")

In [ ]:
X, y = [], []

for img_dir in CATEGORY_LABELS:
    sample_count = len(os.listdir(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir)))
    print(f"Importing category: {img_dir} ({sample_count} samples)")

    for filename in os.listdir(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir)):
        if not (filename.endswith(".jpg") or filename.endswith(".png")): continue

        img = cv2.imread(os.path.join(GRAYSCALE_IMAGE_DIR, img_dir, filename), cv2.IMREAD_GRAYSCALE)
        X.append(img)
        y.append(CATEGORY_LABELS.index(img_dir))

print("------------------------------")
print(f"Data imported. Total samples: {len(X)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from cnn import ResNet, Bottleneck

net = ResNet(Bottleneck, [2, 2, 2, 2], num_classes=len(CATEGORY_LABELS))
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

X_tensor = torch.tensor(X, dtype=torch.float32)  # Add channel dimension
y_tensor = torch.tensor(y, dtype=torch.int16)
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
for epoch in range(5):
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:
            print(f"[{epoch + 1}, {i + 1}] loss: {running_loss / 200:.3f}")
            running_loss = 0.0

print("Finished Training") 

## Import Trained Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from cnn import ResNet, Bottleneck
from cnn import ResNet, Bottleneck


net = ResNet(Bottleneck, [2, 2, 2, 2], num_classes=len(CATEGORY_LABELS), num_channels=1)
net.load_state_dict(torch.load("../models/resnet_food_classifier_epoch_15.pth", weights_only=True))
net.eval()


In [ ]:
import cv2, time
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from IPython.display import Image, display, clear_output

EVAL_IMAGE_DIR_GRAYSCALE = ".data/eval_grayscale"
CATEGORY_LABELS = [
    "not_food",
    "italian_food",
    "japanese_food",
    "meat",
    "seafood",
    "soup",
    "salad",
    "dessert"
]


with torch.no_grad():
    eval_images = sorted(os.listdir(os.path.join(EVAL_IMAGE_DIR_GRAYSCALE)))

    for img_path in eval_images:
        img = cv2.imread(os.path.join(EVAL_IMAGE_DIR_GRAYSCALE, img_path), cv2.IMREAD_GRAYSCALE)
        
        tensor = torch.tensor(np.array(img), dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions
        prediction = net(tensor)
        probabilities = F.softmax(prediction, dim=1).numpy()[0]
        
        print(f"PREDICTED LABELS:")
        for i, prob in enumerate(probabilities):
            print(f"{i+1}. {CATEGORY_LABELS[i].replace('_', ' ').capitalize()}: {prob:.3f}")
        
        print("\nPREDICTION:", CATEGORY_LABELS[np.argmax(probabilities)].replace('_', ' ').capitalize())
        
        display(Image(os.path.join(EVAL_IMAGE_DIR_GRAYSCALE, img_path)))

        time.sleep(0.2)  # Pause for a moment to view the prediction

        correctness = input("Is the prediction correct? (y/n): ").strip().lower()
        clear_output(wait=True)
    
    
    pass